# WiDS Wildfire Deterministic Survival Analog Stack


In [1]:
# WiDS Wildfire Deterministic Survival Analog Stack
import os, sys, warnings, math, random
warnings.filterwarnings('ignore')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
_local_site = '/opt/pyvenv/lib/python3.13/site-packages'
if os.path.isdir(_local_site) and _local_site not in sys.path:
    sys.path.append(_local_site)
import numpy as np
import pandas as pd

SEED = 20260419
np.random.seed(SEED)
random.seed(SEED)
PROB_COLS = ['prob_12h','prob_24h','prob_48h','prob_72h']
HORIZONS = [12,24,48]
DIST_HIT_CUTOFF_M = 5000.0
EPS = 1e-7


def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/widsworldwide-globaldathon26',
        '/kaggle/input/WiDSWorldWide_GlobalDathon26',
        '/kaggle/input/wiDSworldwide_globaldathon26',
        '/kaggle/input/widsworldwide-globaldatathon26',
        '/kaggle/input/widsworldwide-globaldathon-2026',
        '.',
        '/mnt/data',
    ]
    for p in candidates:
        if os.path.isdir(p) and all(os.path.exists(os.path.join(p, f)) for f in ['train.csv','test.csv','sample_submission.csv']):
            return p
    if os.path.isdir('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if {'train.csv','test.csv','sample_submission.csv'}.issubset(set(files)):
                return root
    raise FileNotFoundError('Could not find train.csv, test.csv, sample_submission.csv')

DATA_DIR = find_data_dir()
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
OUTPUT_PATH = os.path.join(OUT_DIR, 'submission.csv')
train = pd.read_csv(os.path.join(DATA_DIR,'train.csv'))
test = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))
sample = pd.read_csv(os.path.join(DATA_DIR,'sample_submission.csv'))


def safe_div(a,b,default=0.0):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float)
    out=np.full(np.broadcast(a,b).shape, default, dtype=float)
    return np.divide(a,b,out=out,where=np.abs(b)>1e-12)


def enrich(df):
    r=df.copy()
    dist=r['dist_min_ci_0_5h'].astype(float).clip(lower=1.0)
    area=r['area_first_ha'].astype(float).clip(lower=0.0)
    per=r['num_perimeters_0_5h'].astype(float)
    dt=r['dt_first_last_0_5h'].astype(float)
    low=r['low_temporal_resolution_0_5h'].astype(float)
    align=r['alignment_abs'].astype(float).clip(0,1)
    close=r['closing_speed_m_per_h'].astype(float)
    close_pos=close.clip(lower=0.0)
    radial=r['radial_growth_rate_m_per_h'].astype(float)
    radial_pos=radial.clip(lower=0.0)
    eff=close_pos+radial_pos
    radius=np.sqrt(area*10000.0/np.pi)
    hour=r['event_start_hour'].astype(float)
    dow=r['event_start_dayofweek'].astype(float)
    mon=r['event_start_month'].astype(float)

    r['dist_km']=dist/1000.0
    r['dist_margin_5km']=(DIST_HIT_CUTOFF_M-dist)/1000.0
    r['log_dist']=np.log1p(dist)
    r['inv_dist']=1.0/(r['dist_km']+0.05)
    r['inside_5km']=(dist<DIST_HIT_CUTOFF_M).astype(float)
    r['inside_3km']=(dist<3000).astype(float)
    r['inside_2km']=(dist<2000).astype(float)
    r['inside_1km']=(dist<1000).astype(float)
    r['log_area']=np.log1p(area)
    r['sqrt_area']=np.sqrt(area)
    r['fire_radius_m']=radius
    r['radius_to_dist']=safe_div(radius,dist)
    r['area_per_dist']=safe_div(area,r['dist_km']+0.05)
    r['log_area_minus_log_dist']=np.log1p(area)-np.log1p(dist)
    r['has_move']=(per>1).astype(float)
    r['one_perim']=(per==1).astype(float)
    r['two_perim']=(per==2).astype(float)
    r['three_plus_perim']=(per>=3).astype(float)
    r['five_plus_perim']=(per>=5).astype(float)
    r['closing_pos']=close_pos
    r['closing_neg']=(-close).clip(lower=0.0)
    r['radial_pos']=radial_pos
    r['eff_speed']=eff
    r['log_eff_speed']=np.log1p(eff)
    r['eta_eff']=np.where(eff>1e-6, dist/eff, 9999.0).clip(0,9999)
    r['log_eta_eff']=np.log1p(r['eta_eff'])
    r['align_eff']=align*eff
    r['align_close']=align*close_pos
    r['threat']=align*eff/np.log1p(dist)
    r['advance_pos']=r['projected_advance_m'].astype(float).clip(lower=0.0)
    r['advance_per_dist']=safe_div(r['advance_pos'],dist)
    r['toward_slope']=(-r['dist_slope_ci_0_5h'].astype(float)).clip(lower=0.0)
    r['toward_accel']=(-r['dist_accel_m_per_h2'].astype(float)).clip(lower=0.0)
    r['cross_abs']=np.abs(r['cross_track_component'].astype(float))
    r['along_pos']=r['along_track_speed'].astype(float).clip(lower=0.0)
    r['along_neg']=(-r['along_track_speed'].astype(float)).clip(lower=0.0)
    r['low_x_dist']=low*r['dist_km']
    r['low_x_area']=low*r['log_area']
    r['low_x_hour']=low*hour
    r['low_x_month']=low*mon
    r['perim_x_align']=per*align
    r['perim_x_eff']=per*eff
    r['dt_x_eff']=dt*eff
    r['hour_sin']=np.sin(2*np.pi*hour/24.0)
    r['hour_cos']=np.cos(2*np.pi*hour/24.0)
    r['dow_sin']=np.sin(2*np.pi*dow/7.0)
    r['dow_cos']=np.cos(2*np.pi*dow/7.0)
    r['month_sin']=np.sin(2*np.pi*(mon-1)/12.0)
    r['month_cos']=np.cos(2*np.pi*(mon-1)/12.0)
    r['summer']=r['event_start_month'].isin([6,7,8]).astype(float)
    r['peak_summer']=r['event_start_month'].isin([7,8]).astype(float)
    r['late_afternoon']=((r['event_start_hour']>=16)&(r['event_start_hour']<=19)).astype(float)
    r['evening_20_22']=((r['event_start_hour']>=20)&(r['event_start_hour']<=22)).astype(float)
    r['night']=((r['event_start_hour']<=6)|(r['event_start_hour']>=22)).astype(float)
    r=r.replace([np.inf,-np.inf],np.nan).fillna(0.0)
    return r


def group_frame(df):
    g=pd.DataFrame(index=df.index)
    g['lowres']=df['low_temporal_resolution_0_5h'].astype(int).astype(str)
    g['perim_bin']=pd.cut(df['num_perimeters_0_5h'].astype(float),[-0.1,1,2,4,8,999],labels=['1','2','3-4','5-8','9+']).astype(str)
    g['dist_bin']=pd.cut(df['dist_min_ci_0_5h'].astype(float),[0,1000,2000,3000,4000,5000],include_lowest=True,labels=['0-1','1-2','2-3','3-4','4-5']).astype(str)
    g['area_bin']=pd.cut(df['area_first_ha'].astype(float),[-0.1,5,20,50,100,200,500,1e9],labels=['a0','a1','a2','a3','a4','a5','a6']).astype(str)
    g['align_bin']=pd.cut(df['alignment_abs'].astype(float),[-0.01,0.001,0.2,0.5,0.8,1.01],labels=['zero','lo','mid','hi','vhi']).astype(str)
    g['hour_bin']=pd.cut(df['event_start_hour'].astype(float),[-0.1,6,12,16,19,21,23],labels=['0-6','7-12','13-16','17-19','20-21','22-23']).astype(str)
    g['month']=df['event_start_month'].astype(int).astype(str)
    g['dow']=df['event_start_dayofweek'].astype(int).astype(str)
    for a,b in [('lowres','perim_bin'),('lowres','area_bin'),('lowres','hour_bin'),('align_bin','perim_bin'),('month','hour_bin')]:
        g[a+'_'+b]=g[a]+'_'+g[b]
    return g


def eb_predict(train_groups,test_groups,y,alpha=10.0):
    y=np.asarray(y,dtype=float)
    global_rate=float(y.mean())
    specs=[
        ('lowres',3.5,alpha*0.65),('perim_bin',2.8,alpha*0.75),('align_bin',2.2,alpha*0.8),
        ('lowres_perim_bin',2.2,alpha*0.55),('dist_bin',1.4,alpha),('area_bin',1.3,alpha),
        ('hour_bin',1.2,alpha),('month',1.0,alpha*1.2),('dow',0.7,alpha*1.2),
        ('lowres_area_bin',1.0,alpha*0.8),('lowres_hour_bin',1.0,alpha*0.8),
        ('align_bin_perim_bin',1.3,alpha*0.8),('month_hour_bin',0.7,alpha)
    ]
    pred=np.zeros(len(test_groups),dtype=float); total=0.0
    for col,w,a in specs:
        tab=pd.DataFrame({'grp':train_groups[col].astype(str),'y':y}).groupby('grp')['y'].agg(['sum','count'])
        rates=(tab['sum']+a*global_rate)/(tab['count']+a)
        pred += w*test_groups[col].astype(str).map(rates).fillna(global_rate).to_numpy(float)
        total += w
    return np.clip(pred/max(total,1e-12),0,1)


def analog_predict(X_train, y, X_test, feature_sets, sigmas=(1.2,1.8,2.7), k_list=(9,17,33)):
    y=np.asarray(y,dtype=float)
    preds=[]
    for feats, mult in feature_sets:
        feats=[f for f in feats if f in X_train.columns]
        if not feats:
            continue
        A=X_train[feats].to_numpy(float)
        B=X_test[feats].to_numpy(float)
        med=np.nanmedian(A,axis=0)
        q75=np.nanpercentile(A,75,axis=0)
        q25=np.nanpercentile(A,25,axis=0)
        scale=q75-q25
        scale=np.where(scale<1e-6, np.nanstd(A,axis=0), scale)
        scale=np.where(scale<1e-6, 1.0, scale)
        A=(A-med)/scale
        B=(B-med)/scale
        D=((B[:,None,:]-A[None,:,:])**2).mean(axis=2)
        for sigma in sigmas:
            W=np.exp(-D/(2*(sigma*mult)**2))
            W=W+1e-12
            preds.append((W@y)/W.sum(axis=1))
        order=np.argsort(D,axis=1)
        for k in k_list:
            k=min(k,A.shape[0])
            rows=np.arange(B.shape[0])[:,None]
            idx=order[:,:k]
            dsel=D[rows,idx]
            w=1.0/(np.sqrt(dsel)+0.05)
            preds.append((w*y[idx]).sum(axis=1)/w.sum(axis=1))
    if not preds:
        return np.full(len(X_test),float(y.mean()))
    P=np.vstack(preds)
    return np.clip(0.55*np.median(P,axis=0)+0.45*np.mean(P,axis=0),0,1)


def logit_rate_shift(p,target,strength):
    p=np.clip(np.asarray(p,dtype=float),EPS,1-EPS)
    target=float(np.clip(target,EPS,1-EPS))
    cur=float(np.clip(p.mean(),EPS,1-EPS))
    z=np.log(p/(1-p))
    shift=math.log(target/(1-target))-math.log(cur/(1-cur))
    return np.clip(1/(1+np.exp(-(z+strength*shift))),EPS,1-EPS)

tr_e=enrich(train)
te_e=enrich(test)
near_train=train['dist_min_ci_0_5h'].astype(float).to_numpy()<DIST_HIT_CUTOFF_M
near_test=test['dist_min_ci_0_5h'].astype(float).to_numpy()<DIST_HIT_CUTOFF_M
raw_near_train=train.loc[near_train].reset_index(drop=True)
raw_near_test=test.loc[near_test].reset_index(drop=True)
Xn=tr_e.loc[near_train].reset_index(drop=True)
Xtn=te_e.loc[near_test].reset_index(drop=True)
time=raw_near_train['time_to_hit_hours'].to_numpy(float)
Gtr=group_frame(raw_near_train)
Gte=group_frame(raw_near_test)

core=['low_temporal_resolution_0_5h','num_perimeters_0_5h','dt_first_last_0_5h','alignment_abs','dist_km','log_area','radial_pos','closing_pos','eff_speed','event_start_hour','event_start_month','event_start_dayofweek']
geo=['dist_km','dist_margin_5km','inside_3km','inside_2km','inside_1km','fire_radius_m','radius_to_dist','area_per_dist','log_area_minus_log_dist']
move=['has_move','one_perim','two_perim','three_plus_perim','five_plus_perim','radial_pos','closing_pos','eff_speed','align_eff','threat','advance_per_dist','toward_slope','toward_accel','perim_x_eff','dt_x_eff']
timef=['hour_sin','hour_cos','dow_sin','dow_cos','month_sin','month_cos','summer','peak_summer','late_afternoon','evening_20_22','night','low_x_hour','low_x_month']
feature_sets=[(core,1.0),(core+geo,1.15),(core+move,1.2),(core+timef,1.25),(core+geo+move+timef,1.5)]

pred=np.zeros((len(test),4),dtype=float)
if near_test.sum()>0:
    near_pred=np.zeros((int(near_test.sum()),4),dtype=float)
    for j,H in enumerate(HORIZONS):
        y=(time<=H).astype(int)
        base=float(y.mean())
        peb=eb_predict(Gtr,Gte,y,alpha=8.0 if H==12 else 12.0)
        pan=analog_predict(Xn,y,Xtn,feature_sets)
        if H==12:
            p=0.52*peb+0.36*pan+0.12*base
            p=logit_rate_shift(p,base,0.30)
            lo,hi=0.055,0.995
        elif H==24:
            p=0.62*peb+0.25*pan+0.13*base
            p=logit_rate_shift(p,base,0.40)
            lo,hi=0.20,0.998
        else:
            p=0.72*peb+0.15*pan+0.13*base
            p=logit_rate_shift(p,base,0.50)
            lo,hi=0.34,0.999
        near_pred[:,j]=np.clip(p,lo,hi)
    near_pred[:,1]=np.maximum(near_pred[:,1],near_pred[:,0])
    near_pred[:,2]=np.maximum(near_pred[:,2],near_pred[:,1])
    near_pred[:,3]=1.0
    pred[near_test,:]=near_pred
pred[~near_test,:]=0.0
pred=np.clip(pred,0,1)
for j in range(1,4):
    pred[:,j]=np.maximum(pred[:,j],pred[:,j-1])

submission=pd.DataFrame({'event_id':test['event_id'].values,'prob_12h':pred[:,0],'prob_24h':pred[:,1],'prob_48h':pred[:,2],'prob_72h':pred[:,3]})
submission=sample[['event_id']].merge(submission,on='event_id',how='left',validate='one_to_one')
assert list(submission.columns)==['event_id','prob_12h','prob_24h','prob_48h','prob_72h']
assert len(submission)==len(sample) and submission['event_id'].is_unique and set(submission['event_id'])==set(sample['event_id'])
vals=submission[PROB_COLS].to_numpy(float)
assert np.isfinite(vals).all() and ((vals>=0)&(vals<=1)).all() and (np.diff(vals,axis=1)>=-1e-12).all()
submission.to_csv(OUTPUT_PATH,index=False)
print('DATA_DIR:',DATA_DIR)
print('near_test_count:',int(near_test.sum()),'far_test_count:',int((~near_test).sum()))
print('saved:',OUTPUT_PATH)
print(submission.head(12).to_string(index=False))
print(submission[PROB_COLS].describe().to_string())


DATA_DIR: /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
near_test_count: 28 far_test_count: 67
saved: /kaggle/working/submission.csv
 event_id  prob_12h  prob_24h  prob_48h  prob_72h
 10662602  0.000000  0.000000  0.000000       0.0
 13353600  0.639495  0.901646  0.948707       1.0
 13942327  0.000000  0.000000  0.000000       0.0
 16112781  0.645700  0.899701  0.947145       1.0
 17132808  0.000000  0.000000  0.000000       0.0
 17445696  0.000000  0.000000  0.000000       0.0
 17599982  0.000000  0.000000  0.000000       0.0
 18750374  0.625073  0.892417  0.944000       1.0
 21365245  0.000000  0.000000  0.000000       0.0
 23634840  0.658611  0.901341  0.948742       1.0
 24461897  0.000000  0.000000  0.000000       0.0
 25353357  0.000000  0.000000  0.000000       0.0
        prob_12h   prob_24h   prob_48h   prob_72h
count  95.000000  95.000000  95.000000  95.000000
mean    0.201644   0.267316   0.280909   0.294737
std     0.315900   0.415838   0.436911   0.458343
min    